# Nhóm 10 – Dự đoán tái nhập viện ở bệnh nhân đái tháo đường (Diabetes 130-US)
**Notebook tổng hợp:** chạy **script của từng thành viên, lấy trực tiếp từ nhánh của người đó**, theo thứ tự từ trên xuống. Các bước nối với nhau bằng file trong `output/`: output của bước trước là input của bước sau. `Runtime → Run all` là chạy cả pipeline.

| Bước | Task | Người | Script (nhánh) | Đọc | Ghi (`output/`) |
|---|---|---|---|---|---|
| 01 | Bỏ thuộc tính định danh | Dũng | `src/01_explore_and_drop_ids.py` (Dung) | `data/diabetic_data.csv` | `step1.csv`, `step1_patient_nbr.csv` |
| 02 | Thống kê khuyết, bỏ cột khuyết > 50% | Sơn | `src/02_missing_stats_and_filter.py` (Son) | `step1.csv` | `step2.csv`, `missing_rate.csv` |
| 03 | Bỏ dòng khuyết < 2%, tạo nhãn, chia 80/20 | Chiến | `src/03_data_preprocessing_splitting.py` (Chien) | `step2.csv`, `step1_patient_nbr.csv` | `train.csv`, `test.csv` |
| 04 | One-Hot (fit trên train) | Dũng | `src/04_one_hot_encoding.py` (Dung) | `train.csv`, `test.csv` | `X_train_onehot.npz`, `X_test_onehot.npz`, `onehot_columns.txt`, `y_train.csv`, `y_test.csv` |
| 05 | PCA 60 thành phần | Sơn | `src/05_pca.py` (Son) | `X_*_onehot.npz`, `onehot_columns.txt` | `X_train_pca.npy`, `X_test_pca.npy`, `pca_eigenvalues.npy`, `pca_components.npy`, `pca_feature_names.txt` |
| 06 | Chuẩn hóa sau PCA, scree plot, top 5 vector riêng | Chiến | `src/06_feature_scaling_pca_analysis.py` (Chien) | `X_*_pca.npy`, `pca_*` | `X_train_pca_scaled.npy`, `X_test_pca_scaled.npy`, `eigenvalues.png` |
| 07a | k-NN tự cài, PCA chưa vs đã chuẩn hóa | Dũng | `src/07a_knn.py` (Dung) | `X_*_pca*.npy`, `y_*.csv` | `knn_results.csv`, `knn_k_curve.png` |
| 07b | Softmax Regression tự cài, chưa vs đã chuẩn hóa | Dũng | `src/07b_softmax_regression.py` (Dung) | `X_*_pca*.npy`, `y_*.csv` | `softmax_results.csv`, `softmax_loss.png` |


**Quy ước:** mọi script chạy từ thư mục gốc dự án, đọc/ghi đúng tên file trong bảng; không đổi tên khi chưa báo cả nhóm. Các bước học từ dữ liệu chỉ fit trên train. `random_state = 42`. Muốn sửa một bước thì sửa script trên nhánh của mình rồi push, chạy lại notebook là cập nhật.

## 0. Cài đặt
Clone cả ba nhánh vào `branches/`. Thư mục làm việc là `branches/Dung` (có dữ liệu gốc); script của Sơn và Chiến được gọi qua `../Son/src/...`, `../Chien/src/...` nên cùng đọc/ghi một thư mục `output/`.

In [11]:
REPO = "https://github.com/chiensondung/HCMUT-AnIntroductiontoMachineLearningandApplication-Project.git"
%cd /content
!rm -rf /content/branches
for b in ["Dung", "Son", "Chien"]:
    !git clone -q -b {b} {REPO} /content/branches/{b}
    !git -C /content/branches/{b} log -1 --format="{b}: %h %an %ad %s" --date=short
%cd /content/branches/Dung
!pip -q install -r requirements.txt -r ../Son/requirements.txt -r ../Chien/requirements.txt

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: cannot change to '/content/branches/Dung': No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: cannot change to '/content/branches/Son': No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directo

## Giai đoạn 1: Khảo sát và xử lý dữ liệu khuyết

### Bước 01: Bỏ thuộc tính định danh *(Dũng)*

In [12]:
!python src/01_explore_and_drop_ids.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file 'src/01_explore_and_drop_ids.py': [Errno 2] No such file or directory


### Bước 02: Thống kê khuyết, bỏ cột khuyết > 50% *(Sơn)*

In [13]:
!python ../Son/src/02_missing_stats_and_filter.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file '../Son/src/02_missing_stats_and_filter.py': [Errno 2] No such file or directory


### Bước 03: Bỏ dòng khuyết < 2%, tạo nhãn, chia train/test 80/20 *(Chiến)*

In [14]:
!python ../Chien/src/03_data_preprocessing_splitting.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file '../Chien/src/03_data_preprocessing_splitting.py': [Errno 2] No such file or directory


## Giai đoạn 2: One-Hot, PCA, chuẩn hóa

### Bước 04: One-Hot Encoding *(Dũng)*

In [15]:
!python src/04_one_hot_encoding.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file 'src/04_one_hot_encoding.py': [Errno 2] No such file or directory


### Bước 05: PCA 60 thành phần trên ma trận tương quan *(Sơn)*

In [16]:
!python ../Son/src/05_pca.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file '../Son/src/05_pca.py': [Errno 2] No such file or directory


### Bước 06: Chuẩn hóa sau PCA, biểu đồ Eigenvalues, top 5 vector riêng *(Chiến)*

In [17]:
!python ../Chien/src/06_feature_scaling_pca_analysis.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file '../Chien/src/06_feature_scaling_pca_analysis.py': [Errno 2] No such file or directory


### Kết quả Giai đoạn 2

In [21]:
from IPython.display import Image, display
!ls -la output
display(Image("output/eigenvalues.png"))

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
ls: cannot access 'output': No such file or directory


FileNotFoundError: No such file or directory: 'output/eigenvalues.png'

FileNotFoundError: No such file or directory: 'output/eigenvalues.png'

<IPython.core.display.Image object>

## Giai đoạn 3: Huấn luyện mô hình — chưa làm

###Bước 07a: Lập trình và tối ưu k-NN (trên dữ liệu chưa chuẩn hoá và đã chuẩn hoá) *(Dũng)*

In [19]:
!python src/07a_knn.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file 'src/07a_knn.py': [Errno 2] No such file or directory


### Bước 07b: Softmax Regression, chưa chuẩn hóa vs đã chuẩn hóa *(Dũng)*


In [20]:
!python src/07b_softmax_regression.py

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file 'src/07b_softmax_regression.py': [Errno 2] No such file or directory


## Giai đoạn 4: Đánh giá tổng hợp — chưa làm